# Model Hubs


In [1]:
from pathlib import Path

import pandas as pd

from neuralzip import Compressor, HFModel
from config import COMPRESSOR_SUITE, MODEL_SUITE

OUTPUT_DIR = Path("outputs/model_hubs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VERIFY_ROUNDTRIP = False
ACTIVE_COMPRESSORS = [
    compressor
    for compressor in COMPRESSOR_SUITE
    if compressor.get("method") in {"zipnn"}
]


/Users/martin/miniconda3/envs/dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Storing


In [3]:
model_results = []
tensor_results = []

for model_dict in MODEL_SUITE:
    hf_model = HFModel(
        model_dict["model_id"],
        dtype=model_dict["default_dtype"],
    )

    for compressor_dict in ACTIVE_COMPRESSORS:
        compressor = Compressor(
            compressor_dict["method"],
            zipnn_method=compressor_dict.get("zipnn_method", "HUFFMAN"),
            verify=VERIFY_ROUNDTRIP,
        )
        result = compressor.compress(hf_model)

        model_results.append(
            {
                "model_name": model_dict["name"],
                "model_id": model_dict["model_id"],
                "parameters_b": model_dict["parameters_b"],
                "dtype": model_dict["default_dtype"],
                "compressor": compressor_dict["name"],
                "raw_bytes": result["raw_bytes"],
                "compressed_bytes": result["compressed_bytes"],
                "compression_ratio": result["compression_ratio"],
                "space_saving": result["space_saving"],
                "encode_seconds": result["encode_seconds"],
                "decode_seconds": result["decode_seconds"],
                "roundtrip_ok": result["roundtrip_ok"],
                "num_tensors": result["num_tensors"],
            }
        )

        for tensor_result in result["tensor_results"]:
            tensor_results.append(
                {
                    "model_name": model_dict["name"],
                    "model_id": model_dict["model_id"],
                    "compressor": compressor_dict["name"],
                    **tensor_result,
                }
            )

    del hf_model


Loading weights: 100%|██████████| 120/120 [00:00<00:00, 1378.99it/s]
[transformers] ResNetModel LOAD REPORT from: microsoft/resnet-18
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.1.weight | UNEXPECTED |  | 
classifier.1.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 282.16it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias 

In [4]:
model_results_df = pd.DataFrame(model_results)
model_results_df["raw_gib"] = model_results_df["raw_bytes"] / 2**30
model_results_df["compressed_gib"] = model_results_df["compressed_bytes"] / 2**30
model_results_df["space_saving_percent"] = 100 * model_results_df["space_saving"]

model_results_df.to_csv(OUTPUT_DIR / "model_storing_results.csv", index=False)
model_results_df


,model_name,model_id,parameters_b,dtype,compressor,raw_bytes,compressed_bytes,compression_ratio,space_saving,encode_seconds,decode_seconds,roundtrip_ok,num_tensors,raw_gib,compressed_gib,space_saving_percent
0,ResNet-18,microsoft/resnet-18,11.7,bfloat16,ZipNN,22353024,14908711,0.666966,0.333034,0.033149,None,None,60,0.020818,0.013885,33.303382
1,BERT Base Uncased,bert-base-uncased,110.0,bfloat16,ZipNN,218964480,145128443,0.662794,0.337206,0.164273,None,None,199,0.203927,0.135161,33.720555


## Tensor-level details


In [5]:
tensor_results_df = pd.DataFrame(tensor_results)
tensor_results_df.to_csv(OUTPUT_DIR / "model_storing_tensor_results.csv", index=False)

tensor_results_df.sort_values(
    ["model_name", "compressor", "raw_bytes"],
    ascending=[True, True, False],
).head(20)


,model_name,model_id,compressor,name,shape,dtype,numel,raw_bytes,compressed_bytes,compression_ratio,encode_seconds
60,BERT Base Uncased,bert-base-uncased,ZipNN,embeddings.word_embeddings.weight,"(30522, 768)",bfloat16,23440896,46881792,30885830,0.658802,0.042433
75,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.0.intermediate.dense.weight,"(3072, 768)",bfloat16,2359296,4718592,3129934,0.663319,0.002733
77,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.0.output.dense.weight,"(768, 3072)",bfloat16,2359296,4718592,3135981,0.664601,0.002773
91,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.1.intermediate.dense.weight,"(3072, 768)",bfloat16,2359296,4718592,3130391,0.663416,0.002081
93,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.1.output.dense.weight,"(768, 3072)",bfloat16,2359296,4718592,3135918,0.664588,0.001905
107,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.2.intermediate.dense.weight,"(3072, 768)",bfloat16,2359296,4718592,3131001,0.663546,0.002110
109,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.2.output.dense.weight,"(768, 3072)",bfloat16,2359296,4718592,3136824,0.664780,0.002134
123,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.3.intermediate.dense.weight,"(3072, 768)",bfloat16,2359296,4718592,3131441,0.663639,0.001819
125,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.3.output.dense.weight,"(768, 3072)",bfloat16,2359296,4718592,3136482,0.664707,0.001844
139,BERT Base Uncased,bert-base-uncased,ZipNN,encoder.layer.4.intermediate.dense.weight,"(3072, 768)",bfloat16,2359296,4718592,3132311,0.663823,0.002024
